In [19]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, T5ForConditionalGeneration,TrainingArguments

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
train_data=pd.read_csv("/content/drive/MyDrive/samsum-train.csv")
test_data=pd.read_csv("/content/drive/MyDrive/samsum-test.csv")
val_data=pd.read_csv("/content/drive/MyDrive/samsum-validation.csv")

# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)
train_data.head()

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."


In [22]:
print(train_data.shape)
print(test_data.shape)

(4000, 3)
(819, 3)


In [23]:
# Data preprocessing

In [24]:
import re

def clean_data(text):
  text=re.sub(r"\r\n"," ",text) #new lines
  text=re.sub(r"\s+"," ",text) #spaces
  text=re.sub(r"<.*?>"," ",text) #html
  text=text.strip().lower()
  return text

In [25]:
train_data["dialogue"] = train_data["dialogue"].fillna("").astype(str).apply(clean_data)
test_data["dialogue"] = test_data["dialogue"].fillna("").astype(str).apply(clean_data)
val_data["dialogue"] = val_data["dialogue"].fillna("").astype(str).apply(clean_data)

In [26]:
#Tokenization (text->numbers)

In [27]:
tokenizer=T5Tokenizer.from_pretrained("t5-base")

In [28]:
#raw data=> tokenized input for fine tuning

def tokenize(data):
  input=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
  target=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

  input["labels"]=target["input_ids"]
  return input

In [29]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [30]:
#input_ids -> token's ids (1 means end of sequence,then 0 are padding)
#attention mask (1-valid tokenid, 0-padded tokenid)
#labels -> summary's ids (1 means end of sequence,then 0 are padding)
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## **Working with our model**

In [31]:
model=T5ForConditionalGeneration.from_pretrained("t5-base")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [32]:
import torch

if torch.backends.mps.is_available():
  device =torch.device("mps")
elif torch.cuda.is_available():
  device =torch.device("cuda")
else:
  device =torch.device("cpu")
print(device)
model.to(device)

cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [33]:
#Training Arguments

training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500 #(in 500 steps model will go from 0 to 5e05(defauls))
)

In [34]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
#train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.341681,0.342586
2,0.344891,0.317111
3,0.310090,0.310762
4,0.285216,0.308508


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
model.save_pretrained("/content/drive/MyDrive/saved_summary_model")
tokenizer.save_pretrained("/content/drive/MyDrive/saved_summary_model")

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("/content/drive/MyDrive/saved_summary_model")

In [ ]:
#Test the code logic for summarization
def summarize_dialogue(dialogue):
  dialogue=clean_data(dialogue)
  #tokenize
  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  )

  #generate summary(in terms of token ids)
  target=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=input["attention_mask"],
      num_beams=4,
      early_stopping=True
  )

  #token id to text(decoding)
  summary=tokenizer.decode(target[0],skip_special_tokens=True)
  return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)